In [4]:
from pathlib import Path
import os

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
ROOT

PosixPath('/Users/v_angelov/GroupProj/TP53-MUTATIONS')

# 1. Download TCGA Data

In [ ]:
# I had already downloaded the data, so I am skipping this step. If you want to download the data, please run the following command:
#!python scripts/00_download_tcga_data.py 

# 2. Build Dataset

In [6]:
!python scripts/01_build_tcga_dataset.py 

Loading phenotype …
  Primary tumor samples: 10,593
Loading expression matrix (this may take ~30 s) …
  Expression after primary-tumor filter: (9701, 20531)
  Filling 3,197,038 NaN values with 0 (undetected genes)
  Dropped 29 numeric-ID columns and 220 zero-variance genes; kept 20,281 gene symbols
Building TP53 labels …
  Collapsing into 'Other' (< 100 samples): ['In-frame indel', 'Other', 'Synonymous']
  TP53 mutant: 3,130
  TP53 wild-type: 6,571
  Mutation type counts:
mutation_type_collapsed
WT            6571
Missense      1949
Nonsense       444
Frameshift     401
Splice         215
Other          121
Saving processed files …

Done. Files written to data/processed/tcga/
  expression_matched.csv.gz : (9701, 20281)
  tp53_labels.csv           : (9701, 4)
  sample_metadata.csv       : (9701, 3)


# 3. Train binary classifier

In [7]:
!python scripts/03_train_binary.py --data-dir data/processed/tcga --tag tcga

top_3000_variable / majority: val ROC-AUC=0.500, F1=0.000
/Users/v_angelov/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
top_3000_variable / logistic_l2: val ROC-AUC=0.868, F1=0.719
/Users/v_angelov/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/v_angelov/.pyenv/versions/3.

# 4. Train multiclass model

In [8]:
!python scripts/04_train_multiclass.py --data-dir data/processed/tcga --tag tcga

top_3000_variable / majority: val macro-F1=0.135
top_3000_variable / logistic_l2: val macro-F1=0.282
top_3000_variable / elastic_net_logistic: val macro-F1=0.279
top_3000_variable / linear_svm: val macro-F1=0.267
top_3000_variable / random_forest: val macro-F1=0.221
top_3000_variable / extra_trees: val macro-F1=0.228
top_3000_variable / hist_gradient_boosting: val macro-F1=0.246
top_3000_variable / mlp: val macro-F1=0.295
tp53_targets / majority: val macro-F1=0.135
tp53_targets / logistic_l2: val macro-F1=0.314
/Users/v_angelov/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
tp53_targets / elastic_net_logistic: val macro-F1=0.261
tp53_targets / linear_svm: val macro-F1=0.336
tp53_targets / random_forest: val macro-F1=0.319
tp53_targets / extra_trees: val macro-F1=0.336
tp53_targets / hist_gradient_boosting: val macro-F1=0.335
tp53_targets / mlp: val

# 5. Train advanced multiclass model

In [10]:
!python scripts/06_train_advanced_multiclass.py --data-dir data/processed/tcga --tag tcga

Loading data …
  Classes: ['Frameshift', 'Missense', 'Nonsense', 'Other', 'Splice', 'WT']
  Class counts:
mutation_type_collapsed
WT            6571
Missense      1949
Nonsense       444
Frameshift     401
Splice         215
Other          121
  Features after selection: 3000

[1/2] Training LightGBM multiclass …
  val macro-F1=0.254, balanced_acc=0.275
  test macro-F1=0.249, balanced_acc=0.268

[2/2] Training One-vs-Rest LightGBM …
  val macro-F1=0.246, balanced_acc=0.267
  test macro-F1=0.243, balanced_acc=0.260

=== Summary ===
              model      split  macro_f1  balanced_accuracy  weighted_f1
lightgbm_multiclass validation  0.254023           0.275192     0.730043
lightgbm_multiclass       test  0.248599           0.267999     0.728322
       lightgbm_ovr validation  0.245815           0.266917     0.723258
       lightgbm_ovr       test  0.242874           0.259804     0.717030


In [14]:
!python scripts/06_train_advanced_multiclass.py \
  --data-dir data/processed/tcga --tag tcga \
  --feature-set p53_pathway

Loading data …
  Classes: ['Frameshift', 'Missense', 'Nonsense', 'Other', 'Splice', 'WT']
  Class counts:
mutation_type_collapsed
WT            6571
Missense      1949
Nonsense       444
Frameshift     401
Splice         215
Other          121
  Feature set: p53_pathway  (60 features)

[1/2] Training LightGBM multiclass …
  val macro-F1=0.362, balanced_acc=0.355
  test macro-F1=0.339, balanced_acc=0.338

[2/2] Training One-vs-Rest LightGBM …
  val macro-F1=0.331, balanced_acc=0.323
  test macro-F1=0.318, balanced_acc=0.313

=== Summary ===
              model      split  macro_f1  balanced_accuracy  weighted_f1
lightgbm_multiclass validation  0.361940           0.354631     0.776206
lightgbm_multiclass       test  0.338985           0.338327     0.770604
       lightgbm_ovr validation  0.331334           0.322704     0.765625
       lightgbm_ovr       test  0.317513           0.312854     0.756551


# 6. Train two-stage multiclass model

In [13]:
!python scripts/07_train_twostage_multiclass.py --data-dir data/processed/tcga --tag tcga

Loading data …
  Mutant samples after filtering: 3130
  Classes: ['Frameshift', 'Missense', 'Nonsense', 'Other', 'Splice']
  Class counts:
mutation_type_collapsed
Missense      1949
Nonsense       444
Frameshift     401
Splice         215
Other          121
  Train: 2190 | Val: 470 | Test: 470

Training models on mutant-only data …
  top_3000_variable / majority: val macro-F1=0.154  bal-acc=0.200
  top_3000_variable / logistic_l2: val macro-F1=0.236  bal-acc=0.237
  top_3000_variable / elastic_net_logistic: val macro-F1=0.247  bal-acc=0.253
  top_3000_variable / linear_svm: val macro-F1=0.218  bal-acc=0.217
  top_3000_variable / random_forest: val macro-F1=0.154  bal-acc=0.200
  top_3000_variable / extra_trees: val macro-F1=0.154  bal-acc=0.200
  top_3000_variable / hist_gradient_boosting: val macro-F1=0.152  bal-acc=0.196
  top_3000_variable / mlp: val macro-F1=0.158  bal-acc=0.211
  top_3000_variable / lightgbm: val macro-F1=0.152  bal-acc=0.197
  tp53_targets / majority: val macro-F